# Batch Evaluate Robot Retarget Results

Run `evaluation/eval_retargeting.py` for each `robot_retarget*.py` variant output directory using the same task settings.

In [33]:
from pathlib import Path
import re
import subprocess
import tempfile
import time

ROOT = Path("/home/cipher/Codes/Academic-Projects/Retarget-whole")
HOLOSOMA = ROOT / "holosoma_retargeting" / "holosoma_retargeting"
EVAL_SCRIPT = HOLOSOMA / "evaluation" / "eval_retargeting.py"
RESULTS_ROOT = HOLOSOMA / "demo_results_total"
DATA_DIR = HOLOSOMA / "demo_data" / "OMOMO_new"
LOG_DIR = RESULTS_ROOT / "batch_eval_logs"

PYTHON = ["conda", "run", "--live-stream", "-n", "robot", "python"]


## Settings

In [34]:
TASK_TYPE = "object_interaction"
DATA_TYPE = "robot_object"
ROBOT = "g1"
TASK_NAME = "sub3_largebox_003"
DATA_FORMAT = "smplh"
OBJECT_NAME = "largebox"
MAX_WORKERS = 1

# Keep this True to evaluate only TASK_NAME even if a result directory contains other .npz files.
EVALUATE_SINGLE_TASK = True

COMMON_EVAL_ARGS = [
    "--data-dir", str(DATA_DIR),
    "--data-type", DATA_TYPE,
    "--robot", ROBOT,
    "--data-format", DATA_FORMAT,
    "--object-name", OBJECT_NAME,
    "--max-workers", str(MAX_WORKERS),
]


## Variants

In [35]:
SCRIPTS = [
    "robot_retarget.py",
    "robot_retarget_etasp.py",
    "robot_retarget_first_order.py",
    "robot_retarget_first_order_etasp.py",
    "robot_retarget_first_order_no_trust_region.py",
    "robot_retarget_first_order_no_trust_region_etasp.py",
    "robot_retarget_laplacian_smooth.py",
    "robot_retarget_laplacian_smooth_etasp.py",
    "robot_retarget_laplacian_smooth_first_order.py",
    "robot_retarget_laplacian_smooth_first_order_etasp.py",
    "robot_retarget_laplacian_smooth_first_order_no_trust_region.py",
    "robot_retarget_laplacian_smooth_first_order_no_trust_region_etasp.py",
    "robot_retarget_laplacian_smooth_no_trust_region.py",
    "robot_retarget_laplacian_smooth_no_trust_region_etasp.py",
    "robot_retarget_no_trust_region.py",
    "robot_retarget_no_trust_region_etasp.py",
]

def variant_name(script_name):
    stem = Path(script_name).stem
    if stem == "robot_retarget":
        return "original"
    return stem.removeprefix("robot_retarget_")

def result_dir_for_variant(variant):
    if TASK_TYPE in {"robot_only", "object_interaction"}:
        data_folder = "omomo"
    elif TASK_TYPE == "climbing":
        data_folder = "mocap_climb"
    else:
        raise ValueError(f"Unsupported TASK_TYPE: {TASK_TYPE}")
    return RESULTS_ROOT / variant / ROBOT / TASK_TYPE / data_folder

VARIANTS = [
    (script_name, variant_name(script_name), result_dir_for_variant(variant_name(script_name)))
    for script_name in SCRIPTS
]

for script_name, variant, res_dir in VARIANTS:
    print(f"{variant:50s} {res_dir}")


original                                           /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/original/g1/object_interaction/omomo
etasp                                              /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/etasp/g1/object_interaction/omomo
first_order                                        /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/first_order/g1/object_interaction/omomo
first_order_etasp                                  /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/first_order_etasp/g1/object_interaction/omomo
first_order_no_trust_region                        /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/first_order_no_trust_re

## Run

In [36]:
METRIC_RE = re.compile(r"^([A-Za-z0-9_]+): mean=([-+0-9.eE]+), std=([-+0-9.eE]+)$")

def parse_metrics(output):
    metrics = {}
    for line in output.splitlines():
        match = METRIC_RE.match(line.strip())
        if not match:
            continue
        name, mean, std = match.groups()
        metrics[f"{name}_mean"] = float(mean)
        metrics[f"{name}_std"] = float(std)
    return metrics

def expected_result_name():
    if DATA_TYPE in {"robot_object", "robot_terrain"}:
        return f"{TASK_NAME}_original.npz"
    if DATA_TYPE == "robot_only":
        return f"{TASK_NAME}.npz"
    raise ValueError(f"Unsupported DATA_TYPE: {DATA_TYPE}")

LOG_DIR.mkdir(parents=True, exist_ok=True)
successes = []
failures = []
rows = []
result_name = expected_result_name()

for script_name, variant, res_dir in VARIANTS:
    result_file = res_dir / result_name

    print(f"\nEvaluating {variant} ({script_name})")
    print(f"result dir: {res_dir}")

    if not res_dir.exists():
        print(f"Missing result directory: {res_dir}")
        failures.append((variant, script_name, "missing result directory", None))
        continue

    if EVALUATE_SINGLE_TASK and not result_file.exists():
        print(f"Missing result file: {result_file}")
        failures.append((variant, script_name, "missing result file", None))
        continue

    tmp_ctx = None
    eval_res_dir = res_dir
    if EVALUATE_SINGLE_TASK:
        tmp_ctx = tempfile.TemporaryDirectory(prefix=f"eval_{variant}_")
        eval_res_dir = Path(tmp_ctx.name)
        (eval_res_dir / result_name).symlink_to(result_file)

    cmd = [*PYTHON, str(EVAL_SCRIPT), "--res-dir", str(eval_res_dir), *COMMON_EVAL_ARGS]

    try:
        start = time.time()
        result = subprocess.run(
            cmd,
            cwd=HOLOSOMA,
            text=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        elapsed = time.time() - start
    finally:
        if tmp_ctx is not None:
            tmp_ctx.cleanup()

    output = result.stdout or ""
    log_path = LOG_DIR / f"{variant}.log"
    log_path.write_text(output)

    print(output)
    print(f"return code: {result.returncode}, time: {elapsed:.1f}s")
    print(f"log: {log_path}")

    if result.returncode != 0:
        failures.append((variant, script_name, result.returncode, elapsed))
        print(f"Failed: {variant}; continuing.")
        continue

    metrics = parse_metrics(output)
    rows.append({
        "variant": variant,
        "script": script_name,
        "res_dir": str(res_dir),
        "elapsed_s": elapsed,
        **metrics,
    })
    successes.append((variant, script_name, elapsed))

print("\nDone.")
print(f"Succeeded: {len(successes)}")
for variant, script_name, elapsed in successes:
    print(f"  OK     {variant:50s} {script_name} ({elapsed:.1f}s)")

print(f"Failed: {len(failures)}")
for variant, script_name, status, elapsed in failures:
    if elapsed is None:
        print(f"  FAIL   {variant:50s} {script_name} ({status})")
    else:
        print(f"  FAIL   {variant:50s} {script_name} (return code {status}, {elapsed:.1f}s)")

if rows:
    try:
        import pandas as pd

        display(pd.DataFrame(rows).sort_values("variant"))
    except ImportError:
        metric_names = sorted({
            key
            for row in rows
            for key in row
            if key not in {"variant", "script", "res_dir", "elapsed_s"}
        })
        for row in sorted(rows, key=lambda item: item["variant"]):
            values = ", ".join(f"{name}={row.get(name, float('nan')):.6f}" for name in metric_names)
            print(f"{row['variant']}: {values}")



Evaluating original (robot_retarget.py)
result dir: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/original/g1/object_interaction/omomo
Found 1 tasks
Loading robot model from:  models/g1/g1_29dof_w_largebox.xml
penetration_duration: mean=0.000000, std=0.000000
penetration_max_depths: mean=0.000000, std=0.000000
sliding_duration: mean=0.000000, std=0.000000
max_toe_sliding_velocities: mean=0.000000, std=0.000000
contact_preservation: mean=1.000000, std=0.000000
opt_cost: mean=0.567960, std=0.000000

return code: 0, time: 5.4s
log: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/batch_eval_logs/original.log

Evaluating etasp (robot_retarget_etasp.py)
result dir: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/etasp/g1/object_interaction/omomo
Found 1 tasks
Loading robot model from:  models/g1/g

,variant,script,res_dir,elapsed_s,penetration_duration_mean,penetration_duration_std,penetration_max_depths_mean,penetration_max_depths_std,sliding_duration_mean,sliding_duration_std,max_toe_sliding_velocities_mean,max_toe_sliding_velocities_std,contact_preservation_mean,contact_preservation_std,opt_cost_mean,opt_cost_std
1,etasp,robot_retarget_etasp.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.049335,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.989796,0.0,2.324009,0.0
2,first_order,robot_retarget_first_order.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.151114,0.035714,0.0,0.048050,0.006044,0.056410,0.0,0.012250,0.001298,0.198980,0.0,-0.975405,0.0
3,first_order_etasp,robot_retarget_first_order_etasp.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.017705,0.045918,0.0,0.032157,0.020634,0.000000,0.0,0.000000,0.000000,0.678571,0.0,-0.248634,0.0
4,laplacian_smooth,robot_retarget_laplacian_smooth.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.439465,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.0,0.567700,0.0
5,laplacian_smooth_etasp,robot_retarget_laplacian_smooth_etasp.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.795634,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.989796,0.0,2.326539,0.0
6,laplacian_smooth_first_order,robot_retarget_laplacian_smooth_first_order.py,/home/cipher/Codes/Academic-Projects/Retarget-...,4.849277,0.000000,0.0,0.000000,0.000000,0.005128,0.0,0.010672,0.000000,0.198980,0.0,0.038447,0.0
7,laplacian_smooth_first_order_etasp,robot_retarget_laplacian_smooth_first_order_et...,/home/cipher/Codes/Academic-Projects/Retarget-...,5.018586,0.005102,0.0,0.012729,0.000000,0.000000,0.0,0.000000,0.000000,0.867347,0.0,-0.222582,0.0
8,laplacian_smooth_no_trust_region,robot_retarget_laplacian_smooth_no_trust_regio...,/home/cipher/Codes/Academic-Projects/Retarget-...,5.313780,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.0,0.567941,0.0
9,laplacian_smooth_no_trust_region_etasp,robot_retarget_laplacian_smooth_no_trust_regio...,/home/cipher/Codes/Academic-Projects/Retarget-...,5.347323,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.989796,0.0,2.326494,0.0
10,no_trust_region,robot_retarget_no_trust_region.py,/home/cipher/Codes/Academic-Projects/Retarget-...,5.539695,0.000000,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,1.000000,0.0,0.568121,0.0


## Frame Curves

This section reads per-frame raw, unweighted `original_lap_cost`, `original_smooth_cost`, and `original_lap_smooth_cost` arrays saved by the retargeter during generation, then plots them directly with Plotly. These fields evaluate each final frame solution under the same unweighted nonlinear laplacian and smooth terms, independent of the solver's internal objective weights.

Only variants that succeeded in the batch evaluation cell are plotted. Delete stale result files and rerun the retargeting scripts after changing the saved cost definition.


In [5]:
FRAME_CURVE_VARIANTS = [variant for _, variant, _ in VARIANTS]
FRAME_CURVE_MAX_FRAMES = None
FRAME_CURVE_REQUIRE_EVAL_SUCCESS = True
FRAME_CURVE_EXPORT_HTML = True
FRAME_CURVE_HTML_PATH = RESULTS_ROOT / "batch_eval_frame_curves.html"
FRAME_CURVE_CSV_PATH = RESULTS_ROOT / "batch_eval_frame_curves.csv"


In [6]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

ORIGINAL_COST_FIELDS = [
    "original_lap_cost",
    "original_smooth_cost",
    "original_lap_smooth_cost",
]


def successful_frame_curve_variants():
    if not FRAME_CURVE_REQUIRE_EVAL_SUCCESS:
        return list(FRAME_CURVE_VARIANTS), []
    if "successes" not in globals():
        return [], [
            (variant, "batch evaluation cell has not been run; success state unknown")
            for variant in FRAME_CURVE_VARIANTS
        ]

    succeeded = {variant for variant, _, _ in successes}
    failed = {variant for variant, _, _, _ in failures} if "failures" in globals() else set()
    variants_to_plot = [variant for variant in FRAME_CURVE_VARIANTS if variant in succeeded and variant not in failed]
    skipped = [
        (variant, "batch evaluation failed or was skipped")
        for variant in FRAME_CURVE_VARIANTS
        if variant not in variants_to_plot
    ]
    return variants_to_plot, skipped


def load_variant_result(res_dir):
    result_file = res_dir / expected_result_name()
    if not result_file.exists():
        return None, result_file
    return np.load(result_file, allow_pickle=True), result_file


def load_original_cost_rows(variant, result_file, data):
    missing = [field for field in ORIGINAL_COST_FIELDS if field not in data.files]
    if missing:
        return [], missing

    n_frames = min(len(data[field]) for field in ORIGINAL_COST_FIELDS)
    if FRAME_CURVE_MAX_FRAMES is not None:
        n_frames = min(n_frames, FRAME_CURVE_MAX_FRAMES)

    rows = []
    for metric in ORIGINAL_COST_FIELDS:
        values = np.asarray(data[metric], dtype=float)[:n_frames]
        for frame, value in enumerate(values):
            rows.append({
                "variant": variant,
                "result_file": str(result_file),
                "frame": int(frame),
                "metric": metric,
                "value": float(value),
            })
    return rows, []


In [7]:
frame_curve_rows = []
frame_curve_missing = []
frame_curve_variants_to_plot, skipped_variants = successful_frame_curve_variants()

frame_curve_missing.extend(skipped_variants)

variant_lookup = {variant: (script_name, res_dir) for script_name, variant, res_dir in VARIANTS}
for variant in frame_curve_variants_to_plot:
    if variant not in variant_lookup:
        frame_curve_missing.append((variant, "not in VARIANTS"))
        continue
    _, res_dir = variant_lookup[variant]
    data, result_file = load_variant_result(res_dir)
    if data is None:
        frame_curve_missing.append((variant, f"missing {result_file}"))
        continue

    print(f"Loading objective curves: {variant} ({result_file})")
    rows, missing_cost_fields = load_original_cost_rows(variant, result_file, data)
    frame_curve_rows.extend(rows)
    if missing_cost_fields:
        frame_curve_missing.append((variant, "stale objective fields: " + ", ".join(missing_cost_fields)))

frame_curve_df = pd.DataFrame(frame_curve_rows)
if frame_curve_rows:
    FRAME_CURVE_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
    frame_curve_df.to_csv(FRAME_CURVE_CSV_PATH, index=False)
    print(f"Frame-curve rows: {len(frame_curve_df)}")
    print(f"CSV: {FRAME_CURVE_CSV_PATH}")
else:
    print("No frame-curve rows produced.")

if frame_curve_missing:
    print("Skipped or stale data:")
    for variant, reason in frame_curve_missing:
        print(f"  {variant}: {reason}")

if frame_curve_rows:
    display(frame_curve_df.head())


Loading objective curves: original (/home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/original/g1/object_interaction/omomo/sub3_largebox_003_original.npz)
Loading objective curves: etasp (/home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/etasp/g1/object_interaction/omomo/sub3_largebox_003_original.npz)
Loading objective curves: first_order (/home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/first_order/g1/object_interaction/omomo/sub3_largebox_003_original.npz)
Loading objective curves: first_order_etasp (/home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/first_order_etasp/g1/object_interaction/omomo/sub3_largebox_003_original.npz)
Loading objective curves: laplacian_smooth (/home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retar

,variant,result_file,frame,metric,value
0,original,/home/cipher/Codes/Academic-Projects/Retarget-...,0,original_lap_cost,0.079218
1,original,/home/cipher/Codes/Academic-Projects/Retarget-...,1,original_lap_cost,0.063842
2,original,/home/cipher/Codes/Academic-Projects/Retarget-...,2,original_lap_cost,0.060173
3,original,/home/cipher/Codes/Academic-Projects/Retarget-...,3,original_lap_cost,0.060640
4,original,/home/cipher/Codes/Academic-Projects/Retarget-...,4,original_lap_cost,0.064108


In [8]:
if frame_curve_df.empty:
    print("No raw original objective curves to plot. Rerun retargeting scripts to regenerate result files with raw original_* cost arrays.")
else:
    fig = px.line(
        frame_curve_df,
        x="frame",
        y="value",
        color="variant",
        facet_row="metric",
        hover_data=["variant", "metric", "frame", "value"],
        title="Raw Unweighted Laplacian and Smooth Terms by Frame",
    )
    fig.update_yaxes(matches=None)
    fig.update_layout(height=900, legend_title_text="variant")
    fig.show()

    if FRAME_CURVE_EXPORT_HTML:
        FRAME_CURVE_HTML_PATH.write_text(fig.to_html(full_html=True, include_plotlyjs="cdn"))
        print(f"HTML: {FRAME_CURVE_HTML_PATH}")


HTML: /home/cipher/Codes/Academic-Projects/Retarget-whole/holosoma_retargeting/holosoma_retargeting/demo_results_total/batch_eval_frame_curves.html
